<a href="https://colab.research.google.com/github/Marlon-Sbardelatti/machine-learning/blob/feature%2Ftrabalho-04/trabalho-04/neural_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Equipe**: 01<br>
> **Estudantes**: Cristina Siewert Jansen, Marlon Sbardelatti e Sofia Sousa Lindner

# **REDES NEURAIS ARTIFICIAIS**

# PARTE 1: Classificação Binária - Diagnóstico de Câncer de Mama

Base de dados: Breast Cancer Wisconsin (Diagnostic) -
https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic

In [ ]:
#bilbiotecas keras e tensorflow
!pip install scikeras==0.13.0

In [ ]:
!pip install tensorflow==2.21.0 scikit-learn==1.9.0

In [3]:
# após instalar as bibliotecas, é necessário reiniciar o notebook
# Para isso, rode esta célula sozinha antes de prosseguir.
exit()

In [1]:
import pandas as pd
import tensorflow as tf
import sklearn
import scikeras

In [2]:
pd.__version__, tf.__version__, sklearn.__version__, scikeras.__version__

('2.2.2', '2.21.0', '1.9.0', '0.13.0')

### Carregar os dados

**1) Carregue a base de dados, faça a divisição de treino e teste (para isso, utilize a função train_test_split do sklearn), como o tamanho da base de teste de 0.25.**

In [4]:
from sklearn.model_selection import train_test_split

data = pd.read_csv('dados_breast.csv')

X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
)

X_train.shape, y_train.shape

((426, 29), (426,))

In [5]:
X_test.shape, y_test.shape

((143, 29), (143,))

### Estrutura da Rede Neural Artificial e Teste

In [6]:
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import cross_val_score
from tensorflow.keras.models import Sequential
import tensorflow as tf
from tensorflow.keras import backend as k

**2) Crie a RNA com as seguintes configurações:**

**a) uma camada de entrada de com 30 neurônios;**

**b) uma camada oculta densa com 16 neurônios. Porque a camada oculta tem essa quantidade de neurônios?**

**c) adicione na camada oculta a função de ativação relu e inicialize os pesos de utilize o Random uniform initializer (https://www.tensorflow.org/api_docs/python/tf/keras/initializers);**

**d) adicione a camada de saíde com a função de ativação sigmoid. Porque foi utilizada esta função para a saída da RNA?**

In [9]:
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.initializers import RandomUniform

ENTRY_LAYER_SIZE = 30
HIDDEN_LAYER_SIZE = 16

# d) Inicializador dos pesos aleatórios da camada oculta
initializer = RandomUniform(minval=0.0, maxval=1.0)

neural_network = Sequential([
    # a) Camada de Entrada
    Input(shape=(ENTRY_LAYER_SIZE,)),

    # b) Camada Oculta
    Dense(units=HIDDEN_LAYER_SIZE, activation='relu', kernel_initializer=initializer),

    # d) Camada de Saída: Sigmoid para classificação binária
    Dense(units=1, activation='sigmoid')
])

classifier = KerasClassifier(
    model=neural_network,
    verbose=0
)

**b) uma camada oculta densa com 16 neurônios. Porque a camada oculta tem essa quantidade de neurônios?**<br>
Porque o número de neurônios da camada oculta é: nº de neurônios de entrada + nº de saídas / 2.<br>
Dessa forma, temos: (30 + 2) / 2 = 32 / 2 = 16.


**d) adicione a camada de saída com a função de ativação sigmoid. Porque foi utilizada esta função para a saída da RNA?**<br>
Porque a classificação é binária, ou seja, só existem duas classes possíveis: 0 (Sem câncer) ou 1 (Com câncer). Sendo assim, a função sigmoide por padrão utiliza o limiar 0.5, o que permite a classificação em exatas duas classes.

**3) Porque utilizamos a classe chamada Sequential para a RNA?**<br>
A classe Sequential permite a definição de camadas sequenciais à rede, o que altera completamente sua arquitetura e comportamento, visto que o resultado de cada camada é utilizado na próxima. Ou seja, para definirmos que os resultados perpassam camada de entrada, oculta e saída sequencialmente, precisamos utilizar o Sequential.

**4) A partir da RNA gerada, explique o que são os valores apresentados na tabela da rede_neural.summary()**<br>
São apresentadas as camadas oculta e de saída da rede, bem como a quantidade de parâmetros e saídas de cada uma. Naturalmente, a camada oculta produz 16 saídas porque contém 16 neurônios, enquanto a camada de saída produz apenas uma: a classe.<br>
Já a coluna de parâmetros exibe quantos parâmetros treináveis existem na camada. Dado que o cálculo dos parâmetros é (nº de entradas x nº de neurônios) + biases, a camada oculta tem 496 pois (31 x 16) + 0 = 496, enquanto a de saída tem 17 pois (16 x 1) + 1 = 17.

In [10]:
neural_network.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 16)             │           496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 513 (2.00 KB)

 Trainable params: 513 (2.00 KB)

 Non-trainable params: 0 (0.00 B)

5) Adicione um otimizador Adam (https://www.tensorflow.org/api_docs/python/tf/keras/optimizers) e especifique a classe loss binário - binary crossentropy (https://www.tensorflow.org/api_docs/python/tf/keras/losses) e a classe metrics para utilizar a métrica de avaliação de acurácia binária (https://www.tensorflow.org/api_docs/python/tf/keras/metrics)

6) Para que servem os otimizadores? Como o otimizador Adam funciona?

7) Depois de estruturados os parâmtros da RNA, utilize a função .fit para fazer o treinamento da rede. Como foi utilizado o otimizador Adam, e ele é baseado na descida do gradiente estocástica, é possível definir a quantidade de registros que serão enviados para a RNA, isto é, em cada batch serão utilizados 10 registros. Quantos batches serão utilizados ao total?

8) Por fim, defina o número de épocas em que ocorre o treinamento igual a 100.

9) Crie uma variável chamada previsoes para realizar a previsão dos dados de teste (X_teste) O resultado da rede deve ser um valor entre 0 e 1. Porque isso acontece?

In [ ]:
previsoes = rede_neural.predict(X_teste)

10) Converta o resultado que está em probabilidade para valores binários com um limiar (treshold) de 0.5.

In [ ]:
previsoes = previsoes > 0.5

In [ ]:
previsoes

9) Explique o resultado da RNA.

In [ ]:
rede_neural.evaluate(X_teste, y_teste)

## Camadas e Otimização da RNA

10) Adicione mais uma camada para oculta densa com 16 neurônio com a função de ativação relu e inicialize os pesos de utilize o Random uniform initializer. Qual o total de parâmetros da RNA agora?

In [ ]:
rede_neural.summary()

11) Abaixo são adicionados os parâmetros do otimizador, que são a taxa de aprendizado e o clipvalue. O que eles fazem?

In [ ]:
otimizador = tf.keras.optimizers.Adam(learning_rate = 0.001, clipvalue = 0.5)
rede_neural.compile(optimizer = otimizador, loss = 'binary_crossentropy', metrics = ['binary_accuracy'])

12) Teste novamente a RNA. Aumentar a quantidade de camadas melhorou ou piorou os resultados? Explique o que aconteceu com a RNA e porque.

In [ ]:
rede_neural.fit(X_treinamento, y_treinamento, batch_size = 10, epochs = 100)

In [ ]:
previsoes = rede_neural.predict(X_teste)

In [ ]:
rede_neural.evaluate(X_teste, y_teste)

## K-Fold Cross Validation

13) Assista o vídeo https://youtu.be/RczbeFs_WbQ?si=f1Yu4LZaGDfZZpbZ para compreender melhor sobre como funciona o k-fold cross validation. Explique como esta rede foi configurada e como é possível chegar no resultado dela. O que é necessário fazer?


14) Calcule também o Desvio Padrão dos resultados para avaliar o modelo. O que é possível concluir com esse resultado?



In [ ]:
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import cross_val_score
from tensorflow.keras.models import Sequential
from tensorflow.keras import backend as k

In [ ]:
def criar_rede():
  k.clear_session()
  rede_neural = Sequential([
      tf.keras.layers.InputLayer(shape=(30,)),
      tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
      tf.keras.layers.Dense(units=16, activation='relu', kernel_initializer='random_uniform'),
      tf.keras.layers.Dense(units=1, activation = 'sigmoid')])
  otimizador = tf.keras.optimizers.Adam(learning_rate = 0.001, clipvalue = 0.5)
  rede_neural.compile(optimizer = otimizador, loss = 'binary_crossentropy', metrics = ['binary_accuracy'])
  return rede_neural

In [ ]:
rede_neural = KerasClassifier(model = criar_rede, epochs = 100, batch_size = 10)

In [ ]:
resultados = cross_val_score(estimator = rede_neural, X = X, y = y, cv = 10, scoring = 'accuracy')

In [ ]:
resultados

## Overfitting e Dropout

14) Como vimos, o modelo anterior sofreu overfitting. Para reduzir esse problema, aplicamos a técnica de regularização conhecida como dropout (artigo oficial: https://www.cs.toronto.edu/~rsalakhu/papers/srivastava14a.pdf e explicação em português: https://www.deeplearningbook.com.br/como-funciona-o-dropout/ ). Aplique o dropout de 20% na primeira e segunda camada oculta. O que acontece com os resultados? E o Desvio Padrão?

In [ ]:
rede_neural = KerasClassifier(model = criar_rede, epochs = 100, batch_size = 10)

In [ ]:
resultados = cross_val_score(estimator = rede_neural, X = X, y = y, cv = 10, scoring = 'accuracy')

In [ ]:
resultados

## Tuning dos Hiperparâmetros (técnicas de ajuste)

Hyperparameter Tuning (Ajuste de Hiperparâmetros) é o processo geral de encontrar os melhores valores para os hiperparâmetros de um modelo de machine learning.
Exemplos de hiperparâmetros:

- Taxa de aprendizado (learning rate)

- Número de árvores em um RandomForest

- Número de camadas ou neurônios em uma rede neural

- Tipo de kernel em um SVM

O tuning pode ser feito por diferentes técnicas, como:

- Grid Search

- Random Search

- Bayesian Optimization, etc.

15) Descreva como a RNA foi configurada para fazer o processo de tuning.

In [ ]:
def criar_rede(optimizer, loss, kernel_initializer, activation, neurons):
  k.clear_session()
  rede_neural = Sequential([
      tf.keras.layers.InputLayer(shape=(30,)),
      tf.keras.layers.Dense(units=neurons, activation=activation, kernel_initializer=kernel_initializer),
      tf.keras.layers.Dropout(rate = 0.2),
      tf.keras.layers.Dense(units=neurons, activation=activation, kernel_initializer=kernel_initializer),
      tf.keras.layers.Dropout(rate = 0.2),
      tf.keras.layers.Dense(units=1, activation = 'sigmoid')])
  rede_neural.compile(optimizer = optimizer, loss = loss, metrics = ['binary_accuracy'])
  return rede_neural

In [ ]:
rede_neural = KerasClassifier(model = criar_rede)

In [ ]:
parametros = {
    'batch_size': [10, 30],
    'epochs': [50, 100],
    'model__optimizer': ['adam', 'sgd'],
    'model__loss': ['binary_crossentropy', 'hinge'],
    'model__kernel_initializer': ['random_uniform', 'normal'],
    'model__activation': ['relu', 'tanh'],
    'model__neurons': [16, 8]
}

In [ ]:
#caso tiver problema com a configuração anterior, testar essa
parametros = {
    'batch_size': [10, 30],
    'epochs': [50],
    'model__optimizer': ['adam'],
    'model__loss': ['binary_crossentropy'],
    'model__kernel_initializer': ['random_uniform', 'normal'],
    'model__activation': ['relu'],
    'model__neurons': [16]
}

In [ ]:
parametros

In [ ]:
grid_search = GridSearchCV(estimator = rede_neural, param_grid = parametros,
                          scoring = 'accuracy', cv = 5)

In [ ]:
grid_search = grid_search.fit(X, y)

In [ ]:
print(grid_search)

In [ ]:
melhores_parametros = grid_search.best_params_
print(melhores_parametros)

In [ ]:
melhor_precisao = grid_search.best_score_
print(melhor_precisao)

16) É possível melhorar ainda mais a curácia da RNA? Como?

## Para salvar o seu modelo de RNA

In [ ]:
classificador.save('classificador_breast.keras')

In [ ]:
classificador_novo = tf.keras.models.load_model('/content/classificador_breast.keras')

In [ ]:
classificador_novo.summary()

# PARTE 2: Redes Neurais e CNN - Identificação de Objetos

Neste trabalho iremos fazer a identificação de um conjunto de dados de imagens de artigos de Zalando chamado Fashion-MNIST, que consiste em um conjunto de treinamento de 60.000 exemplos e um conjunto de teste de 10.000 exemplos. Cada exemplo é uma imagem em tons de cinza 28x28, associada a um rótulo de 10 classes.

Fonte: https://www.tensorflow.org/datasets/catalog/fashion_mnist?hl=pt-br

## **ATIVIDADE:**

**Descrição:**

Nesta atividade, você receberá um código de uma Rede Neural Artificial e uma Rede Neural Convolucional (CNN) implementada. Sua tarefa é analisar o código fornecido e fazer uma interpretação detalhada de cada linha, identificando e explicando as principais etapas, camadas e operações realizadas pela RNA e pela CNN.

**Instruções:**

1. Preencha o código com comentários em cada linha, explicando sua função e propósito.
2. Analise o código cuidadosamente para entender como a CNN é configurada.
3. Identifique e explique as camadas de convolução, camadas de pooling, camadas densas, funções de ativação, tamanhos dos filtros, número de neurônios, etc.
4. Identifique quais técnicas de regularização ou otimização formam utilizadas no código e explique como elas contribuem para o desempenho e a generalização do modelo.
5. Comente sobre a função de ativação utilizada na camada de saída.
6. Descreva os hiperparâmetros e ajustes do modelo, como a taxa de aprendizado, número de épocas de treinamento e o tratamento dos conjuntos de dados de treinamento e teste.
7. Faça uma avaliação descritiva dos resultados das Redes Neurais implementadas.



**Entregável:**

Prepare uma análise detalhada do código fornecido, destacando as principais características da CNN e como ela é configurada para a tarefa em questão. Se necessário, inclua observações sobre o potencial de melhoria ou otimização do modelo.

Esta atividade visa consolidar seu conhecimento sobre CNNs, ajudando a compreender como uma Rede Neural Convolucional é implementada e configurada para tarefas específicas.

In [ ]:
import tensorflow as tf
from tensorflow import keras

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
%matplotlib inline

## Importando base de dados

In [ ]:
fashion_mnist = keras.datasets.fashion_mnist


In [ ]:
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

In [ ]:
print(X_train_full.shape)
print(y_train_full.shape)
print(X_test.shape)
print(y_test.shape)

In [ ]:
# Transformar a escala para que os valores fiquem entre 0 e 1
X_valid, X_train = X_train_full[:5000] / 255.0, X_train_full[5000:] / 255.0
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]

In [ ]:
nomes_classes = ["camisa/top", "calca", "casaco", "vestido", "jaqueta",
               "sandalia", "camiseta", "tenis", "bolsa", "bota"]

In [ ]:
nomes_classes[y_train[5]]

In [ ]:
X_train[5]

In [ ]:
exemplo_n = 5
print(nomes_classes[y_train[exemplo_n]])
plt.imshow(X_train[exemplo_n], interpolation='nearest')
plt.show()

##  Rede Neural Artificial com Keras

In [ ]:
nn = keras.models.Sequential()
nn.add(keras.layers.Flatten(input_shape=[28, 28]))
nn.add(keras.layers.Dense(300, activation="relu"))
nn.add(keras.layers.Dropout(rate=0.1))
nn.add(keras.layers.Dense(100, activation="relu"))
nn.add(keras.layers.Dropout(rate=0.1))
nn.add(keras.layers.Dense(10, activation="softmax"))
nn.summary()

### Compilando e treinando o modelo

In [ ]:
%time
nn.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])
history_nn = nn.fit(X_train, y_train, epochs=5, validation_data=(X_valid, y_valid))

### Visualizar a performance

In [ ]:
pd.DataFrame(history_nn.history).plot(figsize=(12, 8))
plt.grid(True)
plt.gca().set_ylim(0, 1) # set the vertical range to [0-1]
plt.show()

### Avaliar o modelo

In [ ]:
# prints the loss and the accuracy
nn.evaluate(X_test, y_test,verbose=1)

###Realizar uma Previsão

In [ ]:
X_novo = X_test[:4]
y_proba = nn.predict(X_novo)
y_proba

In [ ]:
y_pred = np.argmax(nn.predict(X_novo), axis=-1)
print(y_pred)
print(np.array(nomes_classes)[y_pred])

### Verificar os resultados

In [ ]:
y_novo = y_test[:4]
print(y_novo)
np.array(nomes_classes)[y_pred]

In [ ]:
# example_n = 201
print(nomes_classes[y_pred[2]])
plt.imshow(X_novo[2], interpolation='nearest')
plt.show()

## Treinando CNN com Keras

In [ ]:
cnn = keras.models.Sequential([
keras.layers.InputLayer(input_shape = (28,28,1)),
keras.layers.Conv2D(64,5, activation='relu', padding='same', kernel_initializer='glorot_uniform'),
keras.layers.MaxPool2D(2),
keras.layers.Conv2D(128,3, activation='relu', padding='same'),
keras.layers.Conv2D(128,3, activation='relu', padding='same'),
keras.layers.MaxPool2D(2),
keras.layers.Conv2D(258,3, activation='relu', padding='same'),
keras.layers.Conv2D(258,3, activation='relu', padding='same'),
keras.layers.MaxPool2D(2),
keras.layers.Flatten(),
keras.layers.Dense(128, activation="relu"),
keras.layers.Dropout(rate=0.5),
keras.layers.Dense(64, activation="relu"),
keras.layers.Dropout(rate=0.5),
keras.layers.Dense(10, activation="softmax")])


In [ ]:
cnn.summary()

In [ ]:
print(X_train.shape)
X_train_new = X_train.reshape(X_train.shape[0], X_train.shape[1], X_train.shape[2],1)
X_valid_new = X_valid.reshape(X_valid.shape[0], X_valid.shape[1], X_valid.shape[2],1)
X_test_new = X_test.reshape(X_test.shape[0], X_test.shape[1], X_test.shape[2],1)
print(X_train_new.shape)

In [ ]:
cnn.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])
history_cnn = cnn.fit(X_train_new, y_train, epochs=20, validation_data=(X_valid_new, y_valid))

### Avaliar o modelo

In [ ]:
cnn.evaluate(X_test_new, y_test,verbose=1)

In [ ]:
pd.DataFrame(history_cnn.history).plot(figsize=(12, 8))
plt.grid(True)
plt.gca().set_ylim(0, 1) # set the vertical range to [0-1]
plt.show()